# Predicción de la condición de desocupación en la PEA de Argentina
## Machine Learning con selección de variables optimizada por Algoritmos Genéticos

**Materia:** Algoritmos Genéticos  
**Carrera:** Ingeniería en Sistemas de Información  
**Fuente de datos:** INDEC - EPH Total Urbano - 3er Trimestre 2025

## 1. Introducción

Este notebook desarrolla un modelo predictivo para clasificar personas de la Población Económicamente Activa (PEA) como **ocupadas** o **desocupadas** utilizando características sociodemográficas de la Encuesta Permanente de Hogares (EPH) del INDEC.

Se utiliza un **Algoritmo Genético** para optimizar la selección de variables predictoras y se compara con un modelo base que utiliza todas las variables disponibles.

## 2. Objetivo

Construir un modelo predictivo basado en Random Forest capaz de clasificar personas como ocupadas (0) o desocupadas (1), y evaluar si un Algoritmo Genético puede optimizar la selección de variables predictoras, mejorando o manteniendo el desempeño del modelo.

## 3. Fuente de datos

- **Fuente:** INDEC - Instituto Nacional de Estadística y Censos
- **Encuesta:** Encuesta Permanente de Hogares (EPH) - Total urbano
- **Período:** Tercer trimestre de 2025
- **Archivo:** `personas_tot.urb_3T_2025.txt`
- **Descarga:** https://www.indec.gob.ar/indec/web/Institucional-Indec-BasesDeDatos

## 4. Carga de datos

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Agregar directorio raiz al path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from codigo.configuracion import (
    ARCHIVO_PERSONAS, SEPARADOR_CSV, VARIABLE_ESTADO,
    VALOR_OCUPADO, VALOR_DESOCUPADO, NOMBRE_TARGET,
    NOMBRES_PREDICTORAS, VARIABLES_PREDICTORAS,
    SEMILLA, PARAMETROS_RF, PARAMETROS_AG, FOLDS_CV,
    COLORES, RUTA_GRAFICOS
)

print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')
print(f'Semilla de reproducibilidad: {SEMILLA}')

In [ ]:
# Cargar microdatos de la EPH
datos = pd.read_csv(ARCHIVO_PERSONAS, sep=SEPARADOR_CSV, low_memory=False)
print(f'Datos cargados: {datos.shape[0]:,} filas x {datos.shape[1]} columnas')
datos.head()

## 5. Exploración inicial

In [ ]:
# Distribucion de la variable ESTADO
print('Distribucion de ESTADO (condicion de actividad):')
print()
etiquetas_estado = {
    0: 'Entrevista no realizada',
    1: 'Ocupado',
    2: 'Desocupado',
    3: 'Inactivo',
    4: 'Menor de 10 anios'
}
for valor, conteo in datos['ESTADO'].value_counts().sort_index().items():
    etiqueta = etiquetas_estado.get(valor, f'Codigo {valor}')
    print(f'  {valor} ({etiqueta}): {conteo:,} ({conteo/len(datos)*100:.2f}%)')

In [ ]:
# Explorar variables clave
variables_explorar = ['CH04', 'CH06', 'CH07', 'CH08', 'CH03', 'NIVEL_ED', 'CH09', 'CH15']
for var in variables_explorar:
    info = VARIABLES_PREDICTORAS.get(var, {})
    print(f'\n=== {var}: {info.get("descripcion", var)} ===')
    print(f'Tipo: {info.get("tipo", "desconocido")}')
    print(f'Valores unicos: {datos[var].nunique()}')
    print(f'Valores nulos: {datos[var].isna().sum()}')
    if info.get('tipo') in ['categorica', 'ordinal']:
        print('Distribucion:')
        for valor, conteo in datos[var].value_counts().sort_index().items():
            etiqueta = info.get('codigos', {}).get(valor, '')
            especial = ' [VALOR ESPECIAL]' if valor in info.get('valores_especiales', []) else ''
            print(f'  {valor}: {conteo:,} ({conteo/len(datos)*100:.1f}%) {etiqueta}{especial}')

## 6. Filtrado de la PEA

Se conservan únicamente los registros con `ESTADO = 1` (ocupado) y `ESTADO = 2` (desocupado), que conforman la Población Económicamente Activa.

Se descartan:
- Código 0: Entrevista individual no realizada
- Código 3: Inactivos
- Código 4: Menores de 10 años

In [ ]:
from codigo.carga_datos import filtrar_pea, crear_target

datos_pea, estadisticas_filtrado = filtrar_pea(datos)

## 7. Definición del objetivo

Se crea la variable binaria `desocupado`:
- `ESTADO = 1` (Ocupado) -> `desocupado = 0`
- `ESTADO = 2` (Desocupado) -> `desocupado = 1`

In [ ]:
datos_pea = crear_target(datos_pea)

# Grafico de distribucion
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

etiquetas = ['Ocupado (0)', 'Desocupado (1)']
valores = [(datos_pea['desocupado'] == 0).sum(), (datos_pea['desocupado'] == 1).sum()]
colores = [COLORES['ocupado'], COLORES['desocupado']]

axes[0].bar(etiquetas, valores, color=colores, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribucion de la condicion laboral')
axes[0].set_ylabel('Cantidad')
for i, v in enumerate(valores):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(valores, labels=etiquetas, colors=colores, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporcion de clases')

plt.suptitle('Distribucion de la variable objetivo en la PEA', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Limpieza de datos

Se realiza la limpieza según la documentación oficial del INDEC:
- **CH06 (Edad):** -1 indica menores de 1 anio. No deberían existir en la PEA.
- **CH07 (Estado civil):** Código 9 = Ns/Nr. Se eliminan.
- **CH08 (Cobertura médica):** Código 9 = Ns/Nr. Se eliminan.
- **CH09 (Asistencia escolar):** Código 9 = Ns/Nr. Se eliminan.
- **CH15 (Lugar de nacimiento):** Código 9 = Ns/Nr. Se eliminan.

In [ ]:
from codigo.preprocesamiento_datos import limpiar_datos

datos_limpios, estadisticas_limpieza = limpiar_datos(datos_pea)

## 9. Selección de predictores

Se seleccionan 9 variables sociodemográficas como predictoras candidatas.

**Decisión sobre NIVEL_ED:** Se trata como variable ordinal porque los niveles educativos poseen una jerarquía natural (sin instrucción < primaria incompleta < ... < superior completa). Tratarla como categórica pura descartaría esta información de orden.

In [ ]:
from codigo.preprocesamiento_datos import preparar_predictores

predictores, objetivo = preparar_predictores(datos_limpios)

## 10. Codificación

In [ ]:
from codigo.preprocesamiento_datos import crear_pipeline_codificacion

codificador = crear_pipeline_codificacion(predictores)
print(f'\nVariables categoricas se codificaran con One-Hot Encoding')
print(f'NIVEL_ED ya esta codificado como ordinal')
print(f'CH06 (edad) se conserva como numerica')

## 11. División entrenamiento/prueba

Se divide en 80% entrenamiento y 20% prueba con **división estratificada** para mantener la proporción de clases en ambos conjuntos.

Esto es especialmente importante dado el desbalance (94% ocupados vs 6% desocupados).

In [ ]:
from codigo.preprocesamiento_datos import dividir_datos, obtener_nombres_columnas

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = dividir_datos(predictores, objetivo)

# Codificar (ajustar SOLO con entrenamiento)
X_entrenamiento_cod = codificador.fit_transform(X_entrenamiento)
X_prueba_cod = codificador.transform(X_prueba)

nombres_columnas = obtener_nombres_columnas(codificador, X_entrenamiento)

print(f'\nDimensiones despues de codificacion:')
print(f'  X_entrenamiento: {X_entrenamiento_cod.shape}')
print(f'  X_prueba: {X_prueba_cod.shape}')
print(f'  Variables codificadas: {len(nombres_columnas)}')

## 12. Modelo base

Se entrena un Random Forest con **todas** las variables codificadas como línea de base.

Se utiliza `class_weight='balanced'` para compensar el desbalance de clases.

In [ ]:
from codigo.modelo_base import crear_modelo_base, entrenar_modelo, evaluar_modelo, obtener_importancia_variables

modelo_base = crear_modelo_base()
modelo_base, tiempo_base = entrenar_modelo(modelo_base, X_entrenamiento_cod, y_entrenamiento)

## 13. Evaluación del modelo base

In [ ]:
metricas_base = evaluar_modelo(modelo_base, X_prueba_cod, y_prueba,
                               nombre_modelo='Modelo Base (todas las variables)')
metricas_base['tiempo_entrenamiento'] = tiempo_base
metricas_base['n_variables'] = X_entrenamiento_cod.shape[1]

# Importancia de variables
importancias_base = obtener_importancia_variables(modelo_base, nombres_columnas)

In [ ]:
# Matriz de confusion del modelo base
from codigo.visualizaciones import grafico_matriz_confusion, configurar_estilo
configurar_estilo()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(metricas_base['matriz_confusion'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ocupado (0)', 'Desocupado (1)'],
            yticklabels=['Ocupado (0)', 'Desocupado (1)'],
            linewidths=2, linecolor='white',
            annot_kws={'size': 16, 'fontweight': 'bold'}, ax=ax)
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
ax.set_title('Matriz de confusion - Modelo Base', fontweight='bold')
plt.tight_layout()
plt.show()

## 14. Algoritmo Genético

Se implementa un Algoritmo Genético con la librería DEAP para seleccionar el subconjunto óptimo de variables predictoras.

### Representación cromosómica
Cada individuo es un vector binario: `[1, 0, 1, 1, 0, ...]` donde 1 indica que la variable se utiliza.

### Función de fitness
`fitness = F1_desocupados - penalizacion_por_variables`

Calculado mediante **validación cruzada estratificada de 5 folds** sobre el conjunto de entrenamiento. El conjunto de prueba NO participa.

In [ ]:
from codigo.algoritmo_genetico import ejecutar_algoritmo_genetico

resultado_ag = ejecutar_algoritmo_genetico(
    X_entrenamiento_cod, y_entrenamiento, nombres_columnas
)

## 15. Evolución del Algoritmo Genético

In [ ]:
# Grafico de evolucion del fitness
fig, ax = plt.subplots(figsize=(12, 6))

generaciones = resultado_ag['registro_evolucion']['generacion']
mejor_fitness = resultado_ag['registro_evolucion']['mejor_fitness']
fitness_promedio = resultado_ag['registro_evolucion']['fitness_promedio']

ax.plot(generaciones, mejor_fitness, 'o-', color=COLORES['fitness'],
        linewidth=2, markersize=4, label='Mejor fitness')
ax.plot(generaciones, fitness_promedio, 's--', color='#7f8c8d',
        linewidth=1.5, markersize=3, alpha=0.7, label='Fitness promedio')
ax.fill_between(generaciones, fitness_promedio, mejor_fitness,
                alpha=0.15, color=COLORES['fitness'])

ax.set_xlabel('Generacion')
ax.set_ylabel('F1-score (clase desocupada)')
ax.set_title('Evolucion del fitness del Algoritmo Genetico', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evolucion de la cantidad de variables seleccionadas
fig, ax = plt.subplots(figsize=(12, 5))

n_variables = resultado_ag['registro_evolucion']['n_variables_mejor']
ax.plot(generaciones, n_variables, 'D-', color=COLORES['modelo_opt'],
        linewidth=2, markersize=5)
ax.fill_between(generaciones, n_variables, alpha=0.2, color=COLORES['modelo_opt'])

ax.set_xlabel('Generacion')
ax.set_ylabel('Cantidad de variables seleccionadas')
ax.set_title('Evolucion de la cantidad de variables seleccionadas', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 16. Variables seleccionadas

In [ ]:
print(f'Variables seleccionadas por el AG: {resultado_ag["n_variables_seleccionadas"]} de {resultado_ag["n_variables_total"]}')
print(f'Reduccion: {(1 - resultado_ag["n_variables_seleccionadas"]/resultado_ag["n_variables_total"])*100:.1f}%')
print(f'\nVariables seleccionadas:')
for var in resultado_ag['variables_seleccionadas']:
    print(f'  - {var}')

## 17. Modelo optimizado

Se entrena un Random Forest con **únicamente** las variables seleccionadas por el Algoritmo Genético, usando los mismos parámetros que el modelo base.

In [ ]:
from codigo.evaluacion_modelo import entrenar_modelo_optimizado

metricas_opt, modelo_opt, tiempo_opt = entrenar_modelo_optimizado(
    X_entrenamiento_cod, y_entrenamiento, X_prueba_cod, y_prueba,
    resultado_ag['indices_seleccionados'], nombres_columnas
)

## 18. Evaluación del modelo optimizado

In [ ]:
# Matriz de confusion del modelo optimizado
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(metricas_opt['matriz_confusion'], annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Ocupado (0)', 'Desocupado (1)'],
            yticklabels=['Ocupado (0)', 'Desocupado (1)'],
            linewidths=2, linecolor='white',
            annot_kws={'size': 16, 'fontweight': 'bold'}, ax=ax)
ax.set_xlabel('Clase predicha')
ax.set_ylabel('Clase real')
ax.set_title('Matriz de confusion - Modelo Optimizado (AG)', fontweight='bold')
plt.tight_layout()
plt.show()

## 19. Comparación

Ambos modelos se evalúan sobre **exactamente el mismo conjunto de prueba**.

In [ ]:
from codigo.evaluacion_modelo import comparar_modelos

tabla_comparacion = comparar_modelos(
    metricas_base, metricas_opt,
    tiempo_base, tiempo_opt,
    X_entrenamiento_cod.shape[1], resultado_ag['n_variables_seleccionadas']
)

In [ ]:
# Grafico comparativo
fig, ax = plt.subplots(figsize=(12, 6))

metricas_nombres = ['Accuracy', 'Precision\n(desocupado)', 'Recall\n(desocupado)',
                    'F1-score\n(desocupado)', 'F1 Macro']
valores_base = [metricas_base['accuracy'], metricas_base['precision_desocupado'],
                metricas_base['recall_desocupado'], metricas_base['f1_desocupado'],
                metricas_base['f1_macro']]
valores_opt = [metricas_opt['accuracy'], metricas_opt['precision_desocupado'],
               metricas_opt['recall_desocupado'], metricas_opt['f1_desocupado'],
               metricas_opt['f1_macro']]

x = np.arange(len(metricas_nombres))
ancho = 0.35

barras1 = ax.bar(x - ancho/2, valores_base, ancho, label='Modelo Base',
                 color=COLORES['modelo_base'], edgecolor='white')
barras2 = ax.bar(x + ancho/2, valores_opt, ancho, label='Modelo Optimizado (AG)',
                 color=COLORES['modelo_opt'], edgecolor='white')

for b in barras1:
    ax.text(b.get_x() + b.get_width()/2., b.get_height() + 0.005,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for b in barras2:
    ax.text(b.get_x() + b.get_width()/2., b.get_height() + 0.005,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Valor de la metrica')
ax.set_title('Comparacion de metricas: Modelo Base vs. Modelo Optimizado (AG)', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metricas_nombres)
ax.legend()
ax.set_ylim(0, 1.15)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 20. Interpretación

### Interpretación de la matriz de confusión

- **Verdadero Positivo (VP):** Persona realmente desocupada y clasificada como desocupada. El modelo detectó correctamente la situación.
- **Falso Negativo (FN):** Persona realmente desocupada pero clasificada como ocupada. **Especialmente importante:** el sistema no detectó una situación que se intenta identificar.
- **Falso Positivo (FP):** Persona ocupada clasificada como desocupada.
- **Verdadero Negativo (VN):** Persona ocupada clasificada correctamente.

### Nota sobre causalidad

Las relaciones identificadas por el modelo son de naturaleza **predictiva**, no **causal**. El modelo identifica patrones estadísticos útiles para la predicción, pero no establece que una variable "cause" desocupación.

In [ ]:
# Importancia de variables del modelo base
top_n = 20
nombres_imp = list(importancias_base.keys())[:top_n][::-1]
valores_imp = list(importancias_base.values())[:top_n][::-1]

fig, ax = plt.subplots(figsize=(12, max(6, top_n * 0.35)))
ax.barh(range(len(nombres_imp)), valores_imp, color=COLORES['modelo_base'],
        edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(nombres_imp)))
ax.set_yticklabels(nombres_imp, fontsize=9)
ax.set_xlabel('Importancia (Gini)')
ax.set_title(f'Importancia de las variables (Top {top_n}) - Modelo Base', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 21. Finalidad práctica

El modelo **no** tiene como finalidad simplemente clasificar personas ni tomar decisiones automáticas.

La finalidad práctica es **identificar perfiles o grupos** con mayor probabilidad de desocupación para que organismos públicos puedan orientar recursos hacia:

- Capacitación laboral
- Formación profesional
- Orientación para la búsqueda de empleo
- Programas de inserción laboral
- Seguimiento preventivo

La predicción debe interpretarse como un **indicador de apoyo**. La decisión final corresponde a organismos y responsables humanos.

## 22. Limitaciones

1. El modelo genera predicciones, no certezas.
2. Correlación no implica causalidad.
3. Los datos corresponden al 3er trimestre de 2025; los patrones pueden variar.
4. La EPH tiene características metodológicas específicas.
5. Puede existir sesgo en los datos.
6. Las variables sociodemográficas pueden generar riesgos de discriminación si se usan incorrectamente.
7. El modelo no debería utilizarse para negar derechos o beneficios.
8. La decisión final debe permanecer bajo supervisión humana.

## 23. Conclusiones

Las conclusiones se basan **exclusivamente** en los resultados obtenidos:

In [ ]:
# Resumen de resultados para las conclusiones
print('=== RESUMEN PARA CONCLUSIONES ===')
print(f'\n1. El modelo base obtuvo un F1 de {metricas_base["f1_desocupado"]:.4f} para la clase desocupada')
print(f'   con un Recall de {metricas_base["recall_desocupado"]:.4f}')
print(f'   utilizando {X_entrenamiento_cod.shape[1]} variables')

print(f'\n2. El modelo optimizado obtuvo un F1 de {metricas_opt["f1_desocupado"]:.4f} para la clase desocupada')
print(f'   con un Recall de {metricas_opt["recall_desocupado"]:.4f}')
print(f'   utilizando {resultado_ag["n_variables_seleccionadas"]} variables')

diff_f1 = metricas_opt['f1_desocupado'] - metricas_base['f1_desocupado']
diff_recall = metricas_opt['recall_desocupado'] - metricas_base['recall_desocupado']
reduccion = (1 - resultado_ag['n_variables_seleccionadas'] / resultado_ag['n_variables_total']) * 100

print(f'\n3. Diferencia F1: {diff_f1:+.4f}')
print(f'   Diferencia Recall: {diff_recall:+.4f}')
print(f'   Reduccion de variables: {reduccion:.1f}%')

if diff_f1 > 0.01:
    print(f'\n   CONCLUSION: El AG MEJORO el desempeno del modelo.')
elif diff_f1 > -0.01:
    print(f'\n   CONCLUSION: El AG MANTUVO el desempeno reduciendo variables.')
else:
    print(f'\n   CONCLUSION: El AG REDUJO el desempeno. Esto tambien es un resultado valido.')

print(f'\n4. El Algoritmo Genetico selecciono {resultado_ag["n_variables_seleccionadas"]} variables')
print(f'   de un total de {resultado_ag["n_variables_total"]}, representando una reduccion del {reduccion:.1f}%')

print(f'\n5. Variables seleccionadas por el AG:')
for var in resultado_ag['variables_seleccionadas']:
    print(f'   - {var}')

print(f'\n6. El desbalance de clases (ratio 16:1) representa un desafio importante')
print(f'   para la deteccion de la clase minoritaria (desocupados).')

print(f'\n7. El modelo podria servir como herramienta de apoyo para focalizar')
print(f'   politicas activas de empleo, identificando perfiles con mayor')
print(f'   probabilidad de desocupacion.')